# 04 — Inference & Submission
**SVAMITVA Hackathon — AI-Based Feature Extraction from Drone Images**

This notebook:
1. Loads the best trained checkpoint
2. Runs inference on test orthophotos (sliding window + TTA)
3. Post-processes predictions (morphological cleanup, vectorization)
4. Generates COG and GPKG outputs
5. Evaluates on training data for QC
6. Packages submission

In [ ]:
# ── Cell 1: Setup ────────────────────────────────────────────────
import os, sys, shutil, json, time, glob
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/IITT_AIML')
    LOCAL_ROOT = Path('/content/IITT_AIML')
else:
    DRIVE_ROOT = Path.home() / 'IITT_AIML'
    LOCAL_ROOT = DRIVE_ROOT

sys.path.insert(0, str(LOCAL_ROOT))

if IS_COLAB:
    src_drive = DRIVE_ROOT / 'src'
    src_local = LOCAL_ROOT / 'src'
    if src_drive.exists() and not src_local.exists():
        shutil.copytree(src_drive, src_local)

import torch
import numpy as np
print(f'CUDA: {torch.cuda.is_available()}')

In [ ]:
# ── Cell 2: Find Best Checkpoint ─────────────────────────────────
CKPT_DIR = LOCAL_ROOT / 'checkpoints'
OUTPUT_DIR = LOCAL_ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Restore checkpoints from Drive if needed
if IS_COLAB and not (CKPT_DIR / 'best.pth').exists():
    drive_ckpt = DRIVE_ROOT / 'checkpoints'
    if drive_ckpt.exists():
        CKPT_DIR.mkdir(parents=True, exist_ok=True)
        for f in drive_ckpt.glob('*.pth'):
            shutil.copy2(f, CKPT_DIR / f.name)
            print(f'  Restored: {f.name}')

# Find best checkpoint
best_ckpt = CKPT_DIR / 'best.pth'
if not best_ckpt.exists():
    # Try to find any checkpoint
    ckpts = sorted(CKPT_DIR.glob('*.pth'))
    if ckpts:
        best_ckpt = ckpts[-1]
        print(f'Using latest checkpoint: {best_ckpt.name}')
    else:
        raise FileNotFoundError('No checkpoints found. Run 03_train.ipynb first.')

print(f'Checkpoint: {best_ckpt}')
ckpt = torch.load(best_ckpt, map_location='cpu')
if 'metrics' in ckpt:
    m = ckpt['metrics']
    print(f'  mIoU: {m.get("mean_iou", "?")}  mF1: {m.get("mean_f1", "?")}')

In [ ]:
# ── Cell 3: Load Inferencer ──────────────────────────────────────
from src.config import InferConfig
from src.inference import Inferencer

infer_config = InferConfig(
    tile_size=512,
    overlap=128,           # More overlap for smoother stitching
    use_tta=True,          # Test-time augmentation: hflip, vflip, rot90
    tta_transforms=['hflip', 'vflip'],
    use_ensemble=False,
)

inferencer = Inferencer(
    config=infer_config,
    checkpoint_path=str(best_ckpt),
)
print('Inferencer ready.')

In [ ]:
# ── Cell 4: Run Inference on Test Orthophotos ────────────────────
# Load manifest to find test images
manifest_path = LOCAL_ROOT / 'data_manifest.json'
if not manifest_path.exists() and IS_COLAB:
    manifest_path = DRIVE_ROOT / 'data_manifest.json'

with open(manifest_path) as f:
    manifest = json.load(f)

# Find test orthophotos
test_orthos = [o for o in manifest['working_orthos'] if 'test' in o['state']]

# If no test data, use training data for evaluation
if not test_orthos:
    print('No test orthophotos found. Running on training data for evaluation.')
    # Pick one training orthophoto (smallest) for validation run
    train_orthos = [o for o in manifest['working_orthos'] if 'train' in o['state']]
    if train_orthos:
        # Sort by size (width * height)
        train_orthos.sort(key=lambda o: o['width'] * o['height'])
        test_orthos = [train_orthos[0]]  # Use smallest for quick test
        print(f'  Using: {test_orthos[0]["name"]} for validation')

all_outputs = {}
for ortho_info in test_orthos:
    ortho_path = ortho_info['path']
    fname = Path(ortho_path).stem
    
    t0 = time.time()
    outputs = inferencer.predict_orthophoto(
        ortho_path=ortho_path,
        output_dir=str(OUTPUT_DIR),
        village_name=fname,
    )
    elapsed = time.time() - t0
    
    all_outputs[fname] = outputs
    print(f'  Time: {elapsed/60:.1f} min')

print(f'\nInference complete for {len(all_outputs)} orthophotos.')

In [ ]:
# ── Cell 5: Post-Process & Vectorize ─────────────────────────────
import rasterio
from src.postprocess import postprocess_predictions
from src.config import SEG_CLASSES, InferConfig

for name, output_paths in all_outputs.items():
    seg_path = output_paths.get('segmentation')
    if seg_path is None:
        continue
    
    print(f'\nPost-processing: {name}')
    
    # Read prediction raster
    with rasterio.open(seg_path) as src:
        class_map = src.read(1)
        transform = src.transform
        crs = str(src.crs)
    
    # Full post-processing: cleanup + vectorize + save
    result_paths = postprocess_predictions(
        class_map=class_map,
        transform=transform,
        crs=crs,
        ortho_path=output_paths.get('segmentation', seg_path),
        output_dir=str(OUTPUT_DIR),
        village_name=name,
    )
    
    # Update outputs
    all_outputs[name].update(result_paths)
    
    # Print class statistics
    unique, counts = np.unique(class_map, return_counts=True)
    total = class_map.size
    print('  Class distribution:')
    for cls_id, cnt in zip(unique, counts):
        pct = cnt / total * 100
        cls_name = SEG_CLASSES.get(int(cls_id), f'class_{cls_id}')
        print(f'    {cls_name:12s}: {cnt:10d} px ({pct:5.2f}%)')

print('\nPost-processing complete.')

In [ ]:
# ── Cell 6: Visualize Results ────────────────────────────────────
import matplotlib.pyplot as plt
from src.postprocess import colorize_prediction

for name, output_paths in all_outputs.items():
    seg_path = output_paths.get('segmentation')
    if seg_path is None:
        continue
    
    with rasterio.open(seg_path) as src:
        class_map = src.read(1)
    
    # Show a center crop for visualization
    h, w = class_map.shape
    crop_size = min(2048, h, w)
    cy, cx = h // 2, w // 2
    crop = class_map[cy - crop_size//2:cy + crop_size//2,
                     cx - crop_size//2:cx + crop_size//2]
    
    colored = colorize_prediction(crop)
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(colored)
    ax.set_title(f'{name} — Prediction (center {crop_size}x{crop_size} crop)')
    ax.axis('off')
    
    # Legend
    from src.config import SEG_COLORS
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=[c/255 for c in SEG_COLORS[i]], label=SEG_CLASSES[i])
        for i in SEG_CLASSES if i > 0
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / f'{name}_prediction_viz.png'), dpi=150)
    plt.show()

In [ ]:
# ── Cell 7: Package Submission ───────────────────────────────────
import zipfile

submission_dir = OUTPUT_DIR / 'submission'
submission_dir.mkdir(exist_ok=True)

# Collect all output files
output_files = [
    *OUTPUT_DIR.glob('*_segmentation.tif'),
    *OUTPUT_DIR.glob('*_confidence.tif'),
    *OUTPUT_DIR.glob('*_cleaned.tif'),
    *OUTPUT_DIR.glob('*.gpkg'),
    *OUTPUT_DIR.glob('*_viz.png'),
]

# Create submission zip
zip_path = OUTPUT_DIR / 'submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in output_files:
        zf.write(f, f.name)
        print(f'  Added: {f.name} ({f.stat().st_size/1e6:.1f} MB)')
    
    # Add training curves if available
    curves = LOCAL_ROOT / 'training_curves.png'
    if curves.exists():
        zf.write(curves, 'training_curves.png')

print(f'\nSubmission: {zip_path} ({zip_path.stat().st_size/1e6:.1f} MB)')

# Copy to Drive
if IS_COLAB:
    drive_out = DRIVE_ROOT / 'outputs'
    drive_out.mkdir(parents=True, exist_ok=True)
    shutil.copy2(zip_path, drive_out / 'submission.zip')
    # Also copy individual files
    for f in output_files:
        shutil.copy2(f, drive_out / f.name)
    print(f'All outputs saved to Drive: {drive_out}')

print('\nDone! Download submission.zip for hackathon submission.')